In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [8]:
#Dataset and DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
transform= transforms.Compose([  
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
trainset=CIFAR10(root="./Machine-Learning-Models",train=True, download=True, transform = transform)
testset=CIFAR10(root= "./Machine-Learning-Models",train=False, download=True, transform = transform)

In [9]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./Machine-Learning-Models
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [10]:
testset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./Machine-Learning-Models
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [11]:
trainloader= DataLoader(trainset, batch_size=64, shuffle=True)
testloader=DataLoader(testset, batch_size=64)

In [21]:
# CNN Builder
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.con_layers=nn.Sequential(
            nn.Conv2d(3, 32 ,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel 2 and padding 2
    
            nn.Conv2d(32, 64 ,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel 2 and padding 2
    
            nn.Conv2d(64, 128 ,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) #kernel 2 and padding 2
            
        )
    
        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
    
            nn.Linear(256,10)
        )

    def forward(self,x):
        x=self.con_layers(x)
        x= x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x

In [22]:
model = CNN()

In [23]:
criterion= nn.CrossEntropyLoss()
optimiser=optim.Adam(model.parameters())

In [27]:
# Training CNN
epochs=10

for i in range(epochs):
    training_loss=0
    for images, labels in trainloader:
        optimiser.zero_grad()
        output=model.forward(images)
        loss=criterion(output, labels)
        loss.backward()
        optimiser.step()

        training_loss +=loss.item()
    print(f"Epochs = {i+1}/{epochs} & trainig loss= {training_loss/len(trainloader)}")

Epochs = 1/10 & trainig loss= 0.09601739245940887
Epochs = 2/10 & trainig loss= 0.09813408210368642
Epochs = 3/10 & trainig loss= 0.08652292459290903
Epochs = 4/10 & trainig loss= 0.07559480251746771
Epochs = 5/10 & trainig loss= 0.07378765460118042
Epochs = 6/10 & trainig loss= 0.07259856094680656
Epochs = 7/10 & trainig loss= 0.06866873722940879
Epochs = 8/10 & trainig loss= 0.06890941497064708
Epochs = 9/10 & trainig loss= 0.06470319541091399
Epochs = 10/10 & trainig loss= 0.06055446942051029


In [29]:
# Evaluate our model
correct_label=0
total_label= 0
model.eval()
with torch.no_grad():
    for images, labels in testloader:
        output= model.forward(images)
        _, predicted=torch.max(output,1)

        correct_label+= (predicted==labels).sum().item()
        total_label+=labels.size(0)

print(f"Accuracy = {correct_label/total_label*100}")
        

Accuracy = 74.55000000000001
